# KAT-VAD v4 — **Stage A: hooked V2-S heads on T2 → head selection → G1a** (decision gate V4-1)

Pre-registration: `core/docs/v4/PREREG.md` (I1–I6, S1–S14, G1a) — it must be in the upload and **committed before this
notebook runs**. Proposal: `core/docs/v4/KAT VAD v4 architecture proposal.md` rev 3. Plan: `.project/plans/katvad-v4-resh.md`.

| step | what | GPU? |
|---|---|---|
| 0 | setup, deps; refuses an upload without the v4 tools / PREREG | — |
| 1 | preflight: T2 splits (1,272 train / 219 val), V2-S cache manifest + rows == CLIP rows | no |
| 2 | **hooked V2-S pass over every T2 source** (72 heads, token mean; pooled `u` checked per row, I4) | **yes** |
| 3 | gate I4: every source written, every row cosine ≥ 0.999 | no |
| 4 | **`select_heads`** (S1–S13) → `head_selection.{json,md}` | no (CPU, ~10–30 min) |
| 5 | **STOP — commit `head_selection.json` locally, paste its sha256 (S14)** | — |
| 6 | **`v4_gate g1a`** → `g1a_readout.{json,md}` → **PASS = go to Stage B, FAIL = stop V4** | no (CPU) |

Reads T2-train and T2-val only (T2-test sources are extracted for later stages but never scored here). Nothing reads
DoTA. Every step resumes: re-run the cell after a disconnect.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import sys
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read: data/, cache/ (CLIP, VideoMAE)
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # write: kat-vad/ (code), cache/v4, outputs/v4
os.environ['PROJECT_ROOT']       = DRIVE_V2
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_V2}/outputs'

os.environ['REPO'] = f'{DRIVE_V2}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v4 there'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
assert hasattr(constants, 'V4_STABILITY_MIN_JACCARD'), 'this upload predates v4 Stage A -- re-upload branch v4'
for tool in ('extract_head_features', 'select_heads', 'v4_gate'):
    assert (REPO / f'core/tools/{tool}.py').is_file(), f'upload has no core/tools/{tool}.py'
assert (REPO / 'core/models/videomae_heads.py').is_file(), 'upload has no core/models/videomae_heads.py'
PREREG = REPO / 'core/docs/v4/PREREG.md'
assert PREREG.is_file() and 'S14' in PREREG.read_text(encoding='utf-8'), 'upload has no v4 PREREG'

DATASET = constants.DADA_ORIGIN_DATASET                  # 'DADA2000_orig'
ENC     = constants.V4_ENCODER                           # vit_s_k710_dl_from_giant
STRIDE  = constants.FRAME_STRIDE                         # 8: K's geometry for T2
DIM, DEPTH, NHEADS = constants.VIDEOMAE_ARCH[ENC]        # 384, 12, 6

T2       = constants.DATA_ROOT / DATASET
CENSUS   = T2 / 'counts' / 'resolved_census.json'
ORIG     = constants.DATA_ROOT / 'DADA2000Origin' / constants.DADA_ORIGIN_ROOT_DIRNAME
ANNOT    = ORIG / constants.DADA_ORIGIN_ANNOTATION_FILENAME
CLIP_T2  = constants.CLIP_CACHE_DIR / DATASET
VIDEO    = constants.VIDEO_CACHE_DIR
HEADS    = Path(DRIVE_V2) / 'cache' / 'v4' / 'heads' / ENC / f'{DATASET}_s{STRIDE}_squash'
OUT      = constants.OUTPUT_ROOT / 'v4' / 'REPORTS' / 'stage_a'
LOCAL    = Path('/content/v4')
for p in (HEADS, OUT, LOCAL):
    p.mkdir(parents=True, exist_ok=True)
for name in ('T2', 'CENSUS', 'ANNOT', 'CLIP_T2', 'HEADS', 'OUT'):
    print(f'{name:8s}', globals()[name])

In [ ]:
%%bash
apt-get -qq install -y p7zip-full
pip install -q "transformers==4.56.*" av einops faiss-cpu xlrd
python -c "import transformers, av, einops, faiss; print('transformers', transformers.__version__)"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- step 2 needs one'
df -h /content | tail -1

In [ ]:
import hashlib
import json
import shutil
import subprocess
import time

import numpy as np

ENV = dict(os.environ)


def run(cmd, log=None):
    # Stream a child's output here (and to ``log``); raise with its tail on failure.
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')

## 1. Preflight — splits and the V2-S cache the row check compares against (I3–I5), **before** any GPU time

In [ ]:
from core.tools import build_v2_inputs, e2d_probe, extract_video_features
from core.tools.crn_select import t2_sources

meta = json.loads((T2 / constants.META_FILENAME).read_text())
val, train = t2_sources(meta)
assert (len(train), len(val)) == (1272, 219), (len(train), len(val))
sources = build_v2_inputs.t2_all_sources(T2)                 # train + T2-val + test: one pass for every stage
census = json.loads(CENSUS.read_text())
assert ANNOT.is_file(), f'missing {ANNOT}'

S_DIR = e2d_probe.t2_video_dir(VIDEO, ENC)                   # v2's V2-S cache (A3 / A3-ign's input)
S_MANIFEST = extract_video_features.build_manifest(ENC, STRIDE, constants.VIDEOMAE_WEIGHTS_SHA256[ENC])
e2d_probe.check_manifest(S_DIR, S_MANIFEST)
bad = [v for v in sources
       if len(np.load(S_DIR / f'{v}.npy', mmap_mode='r')) != len(np.load(CLIP_T2 / f'{v}.npy', mmap_mode='r'))]
assert not bad, f'{len(bad)} sources: V2-S rows != CLIP rows: {bad[:5]}'
print(f'{len(sources)} T2 sources ({len(train)} train, {len(val)} T2-val, rest test); V2-S cache == I3 geometry')

## 2. Hooked V2-S pass over **every** T2 source (GPU, resumable; I1–I4)

Same pipeline and geometry as v2's V2-S cache (`p4_s_full_t2_d6.ipynb` §2): 16 frames every 3rd at the assumed 30 fps,
causal, squash 224², stride 8. Writes `(T, 72·64)` float32 per source to `Thesis-V2/cache/v4/heads/…` and the per-row
cosine of the pooled `u` vs the cached row to `ucos/`. A shard whose ids are all written is skipped.
**Re-run this cell after a disconnect.**

In [ ]:
SHARD, BATCH = 60, 32
root, images = constants.DADA_ORIGIN_ROOT_DIRNAME, constants.DADA_ORIGIN_IMAGES_SUBDIR
assert (ORIG / 'DADA2000.zip').is_file(), f'no DADA2000.zip under {ORIG}'
todo = [v for v in sources if not (HEADS / f'{v}.npy').exists()]
print(f'{len(todo)} of {len(sources)} T2 sources to encode')


def type_video(vid):
    t, v = vid.split('_')                      # t{type:02d}_v{video:03d}
    return int(t[1:]), int(v[1:])


started = time.monotonic()
for k in range(0, len(todo), SHARD):
    shard, tag = todo[k:k + SHARD], f'{k // SHARD:03d}'
    if all((HEADS / f'{v}.npy').exists() for v in shard):
        continue
    frames, farm = LOCAL / 'frames', LOCAL / 'flat'
    shutil.rmtree(frames, ignore_errors=True)
    shutil.rmtree(farm, ignore_errors=True)
    farm.mkdir(parents=True)
    patterns = [f'{root}/{t}/{v:03d}/{images}/*' for t, v in map(type_video, shard)]
    print(f'--- shard {tag}: {len(shard)} sources, {shutil.disk_usage("/content").free / 2**30:.1f} GiB free')
    run(['7z', 'x', ORIG / 'DADA2000.zip', f'-o{frames}', '-y', '-bso0', '-bsp0', *patterns])
    for vid in shard:                          # C26: link each source's images dir under its unique id
        t, v = type_video(vid)
        src = frames / root / str(t) / f'{v:03d}' / images
        n = len(list(src.glob('*')))
        if n != census[vid]:                   # C10: a folder is not evidence of data
            raise RuntimeError(f'{vid}: {n} images extracted, census says {census[vid]}')
        (farm / vid).symlink_to(src.resolve(), target_is_directory=True)
    shard_ids = LOCAL / f'ids_{tag}.txt'
    shard_ids.write_text('\n'.join(shard) + '\n')
    run([sys.executable, '-m', 'core.tools.extract_head_features',
         '--frames-dir', farm, '--ids-file', shard_ids, '--dataset', DATASET,
         '--encoder', ENC, '--stride', STRIDE, '--batch-size', BATCH, '--device', 'auto',
         '--reference-dir', S_DIR, '--output-dir', HEADS], log=LOCAL / 'extract_heads.log')
    shutil.rmtree(frames, ignore_errors=True)
    shutil.rmtree(farm, ignore_errors=True)
    print(f'    encoded {sum((HEADS / f"{v}.npy").exists() for v in sources)}/{len(sources)} | '
          f'{(time.monotonic() - started) / 60:.1f} min')
EXTRACT_MIN = (time.monotonic() - started) / 60
print(f'hooked pass: {EXTRACT_MIN:.0f} min this session')

## 3. Gate I4 — every source written, rows == CLIP rows, every pooled-`u` row cosine ≥ 0.999

In [ ]:
from core.tools import extract_head_features as ehf

manifest = json.loads((HEADS / constants.VIDEO_MANIFEST_FILENAME).read_text())
assert (manifest['depth'], manifest['heads'], manifest['head_dim']) == (DEPTH, NHEADS, DIM // NHEADS), manifest
assert manifest['stride'] == STRIDE and manifest['encoder'] == ENC
bad = ehf.ucos_failures(HEADS, sources)
assert not bad, f'I4 FAILED for {len(bad)} sources (−1 = missing): {dict(list(bad.items())[:10])}'
lengths = [v for v in sources
           if len(np.load(HEADS / f'{v}.npy', mmap_mode='r')) != len(np.load(CLIP_T2 / f'{v}.npy', mmap_mode='r'))]
assert not lengths, f'{len(lengths)} sources: head rows != CLIP rows: {lengths[:5]}'
low = min(float(np.load(ehf.ucos_path(HEADS, v)).min()) for v in sources)
print(f'I4 PASS: {len(sources)} sources, min row cosine {low:.6f} (bar {constants.V4_U_COS_MIN})')

## 4. Head selection (S1–S13, CPU; T2-train only)

In [ ]:
run([sys.executable, '-m', 'core.tools.select_heads', '--t2-dir', T2, '--heads-dir', HEADS, '--out-dir', OUT],
    log=LOCAL / 'select.log')
print((OUT / constants.V4_SELECTION_MD).read_text())
SEL_FILE = OUT / constants.V4_SELECTION_FILENAME
print('sha256', hashlib.sha256(SEL_FILE.read_bytes()).hexdigest())

## 5. **STOP — commit the selection before any probe reads it (S14)**

1. Download `Thesis-V2/outputs/v4/REPORTS/stage_a/head_selection.json` (and `.md`).
2. Locally: put it at `outputs/v4/REPORTS/stage_a/head_selection.json` and **commit it** (Claude can do this on your
   word).
3. Paste the sha256 printed above into `SELECTION_SHA256` below. The G1a cell refuses to run otherwise.

If S10 printed **FAIL**, G1a will FAIL too (the stability check is part of G1a.2) — run it anyway: the read-out is the
record.

In [ ]:
SELECTION_SHA256 = ''   # <- paste after committing head_selection.json
assert len(SELECTION_SHA256) == 64, 'commit head_selection.json, then paste its sha256 (S14)'
assert hashlib.sha256(SEL_FILE.read_bytes()).hexdigest() == SELECTION_SHA256, 'the file on Drive changed since'

## 6. G1a — T2-train → T2-val probes (CPU, ~30–60 min: 32 logistic fits)

In [ ]:
run([sys.executable, '-m', 'core.tools.v4_gate', 'g1a', '--t2-dir', T2,
     '--annotation', ANNOT, '--census', CENSUS, '--clip-dir', CLIP_T2, '--video-dir', S_DIR,
     '--heads-dir', HEADS, '--selection', SEL_FILE, '--selection-sha256', SELECTION_SHA256,
     '--out-dir', OUT], log=LOCAL / 'g1a.log')
print((OUT / constants.V4_G1A_MD).read_text())

## 7. What to paste back

`head_selection.md` and `g1a_readout.md` (plus the `.json` files, small). **G1a PASS → Stage B** (Claude builds the
CAP-dev + DF8-window extraction, G1b, G1c). **G1a FAIL → V4 stops at Stage A** (G1a.5): a bounded negative, reported.